# Ch 13. 埋め込み — 解答

> このノートブックは5問すべての厳密な解答と再現可能な検証を含む。

## 問題 1

**問題**: Skip-gram CBOW , .

### 厳密な解説

制御する要因: **objective: Skip-gram center-to-context versus CBOW context-to-center**。  測定指標: **number and shape of training examples**。  解釈と限界: The same windowed corpus yields multiple center-context pairs for Skip-gram and one aggregated context per center for CBOW; the cell constructs both explicitly.


In [ ]:
import torch
corpus='king queen man woman king man'.split(); sg=[]; cb=[]
for i,w in enumerate(corpus):
 ctx=[corpus[j] for j in range(max(0,i-1),min(len(corpus),i+2)) if j!=i]; sg += [(w,c) for c in ctx]; cb.append((tuple(ctx),w))
print({'skipgram_examples':len(sg),'cbow_examples':len(cb),'skipgram_sample':sg[:3],'cbow_sample':cb[:3]}); assert all(len(x)==2 for x in sg)


## 問題 2

**問題**: $K = 1, 5, 15$ 学習 結果 比較.

### 厳密な解説

制御する要因: **negative samples K: 1, 5, 15**。  測定指標: **loss estimate and gradient norm for one positive pair**。  解釈と限界: Matched embeddings and negatives isolate K. More negatives add logistic terms and computation; one stochastic step cannot establish downstream embedding quality.


In [ ]:
import torch
torch.manual_seed(130); center=torch.randn(8,requires_grad=True); pos=torch.randn(8); negatives=torch.randn(15,8)
for K in (1,5,15):
 center.grad=None; loss=-torch.nn.functional.logsigmoid(center@pos)-torch.nn.functional.logsigmoid(-(negatives[:K]@center)).sum(); loss.backward(); print({'K':K,'loss':loss.item(),'center_grad_norm':center.grad.norm().item()})


## 問題 3

**問題**: GloVe 関数 $f(x) = (x/x_{\max})^\alpha$ if $x < x_{\max}$ else 1 .

### 厳密な解説

制御する要因: **co-occurrence count x**。  測定指標: **GloVe weight f(x)**。  解釈と限界: The power law downweights rare/noisy pairs, reaches one at x_max, and caps frequent pairs so they cannot dominate solely by count.


In [ ]:
import torch
xmax=100.; alpha=.75
for x in (1.,5.,20.,100.,500.):
 f=(x/xmax)**alpha if x<xmax else 1.; print({'count':x,'weight':f}); assert 0<f<=1


## 問題 4

**問題**: Word2Vec 結果 .

### 厳密な解説

制御する要因: **analogy vector arithmetic**。  測定指標: **cosine similarity to every candidate**。  解釈と限界: The synthetic embedding encodes gender and royalty as independent axes, so king-man+woman points exactly to queen. This validates the computation, not real-corpus semantic quality.


In [ ]:
import torch
E={'man':torch.tensor([0.,0.]),'woman':torch.tensor([0.,1.]),'king':torch.tensor([1.,0.]),'queen':torch.tensor([1.,1.]),'apple':torch.tensor([-1.,.2])}; q=E['king']-E['man']+E['woman']; scores={w:torch.nn.functional.cosine_similarity(q,E[w],dim=0).item() for w in E if w not in ('king','man','woman')}; print({'query':'king-man+woman','scores':scores,'answer':max(scores,key=scores.get)}); assert max(scores,key=scores.get)=='queen'


## 問題 5

**問題**: $\sqrt{d}$ Attention .

### 厳密な解説

制御する要因: **embedding scale: 1 versus sqrt(d)**。  測定指標: **embedding variance and attention-logit variance**。  解釈と限界: If embedding components have variance about 1/d, multiplying by sqrt(d) restores unit component variance before residual/attention projections. The cell measures that second-moment effect.


In [ ]:
import torch
torch.manual_seed(131); d=64; e=torch.randn(20000,d)/d**.5
for scale in (1.,d**.5):
 x=e*scale; q=x@torch.randn(d,d)/d**.5; k=x@torch.randn(d,d)/d**.5; logits=(q*k).sum(1)/d**.5; print({'scale':scale,'embedding_variance':x.var().item(),'logit_variance':logits.var().item()})
